# Aquaplanet with a customized initial condition

Everything about this run is the shipped `aquaplanet-slab` configuration except one field of the initial carry. The model itself comes through `jem.configurations.load` -- the recipe door onto that validated configuration (issue #131) -- rather than composing Hydra by hand: what this notebook demonstrates (perturbing an initial condition) has nothing to do with how the coupled model was assembled, so the assembly stays one line.

In [ ]:
from pathlib import Path

import jax.numpy as jnp

from jem import configurations, plot, replace_field, run_chunked

output_dir = (Path("output") / "01-02_customized_initial_condition").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

## Build the shipped model

What `exp.coupler` is built from is entirely readable from `exp.config` -- a plain dict, no Hydra object in sight -- e.g. which ocean and how it is forced:

In [ ]:
# The recipe door: the SAME validated `aquaplanet-slab` configuration
# `python -m jem.main +configuration=aquaplanet-slab` builds, with no
# Hydra in this notebook -- see `jem/configurations.py` (issue #131).
exp = configurations.load("aquaplanet-slab")
coupler = exp.coupler
coupler

In [ ]:
# Every choice this configuration pins is a plain Python value here --
# no `+configuration=aquaplanet-slab` needed to find out what it built.
{"ocean": exp.config["ocean"], "seaice": exp.config["seaice"]}

## Change one initial condition

`Coupler.initialize()` returns a `CoupledCarry`; `replace_field` returns a new one with a single `"component.section.field"` replaced, so the original is untouched.

In [ ]:
carry = coupler.initialize()
grid = coupler.components["ocn"].grid
bump = 5 * jnp.sin(2 * grid.longitude_radian) * jnp.cos(grid.latitude_radian) ** 3
sst = carry.components["ocn"]["state"].sea_surface_temperature
carry = replace_field(carry, "ocn.state.sea_surface_temperature", sst + bump)

In [ ]:
import numpy as np
import xarray as xr

# 2-D lat/lon in degrees, straight from the grid -- map_plot handles
# 2-D auxiliary coordinates the same way it does the displaced-pole
# ocean's, so this needs no separate 1-D-vs-2-D case here either.
perturbed_sst = xr.DataArray(
    sst + bump - 273.15,
    dims=("lon_index", "lat_index"),
    coords={
        "lon": (("lon_index", "lat_index"), np.degrees(grid.longitude_radian)),
        "lat": (("lon_index", "lat_index"), np.degrees(grid.latitude_radian)),
    },
    name="sea_surface_temperature",
)
plot.map_plot(perturbed_sst, title="Perturbed initial sea surface temperature [°C]")

## Run it

In [ ]:
result = run_chunked(coupler, total_time=30, chunk=30,
   initial_carry=carry, output_dir=str(output_dir), subsample=3,
   checkpoint_path=None)

In [ ]:
import matplotlib.pyplot as plt

ocn = plot.open_output(output_dir, "ocn")

fig, ax = plt.subplots()
sst_final = ocn["sea_surface_temperature"].isel(time=-1) - 273.15
plot.map_plot(sst_final, ax=ax, title="Final sea surface temperature [°C]")

fig, ax = plt.subplots()
plot.area_mean(ocn["sea_surface_temperature"]).plot(ax=ax)
ax.set_ylabel("Area-mean SST [K]")